# Pipeline test for the KuKi LLM experiments

Runs every script end-to-end on **synthetic aggregate files** (same format as
`kuki_ru_aggregate.jsonl`) with **mock models** (random but schema-valid answers). No GPU needed.
Loading and splitting use exactly the same scripts as the real data. Everything is written to
`<repo>/test_run/`; real data in `<repo>/data/` is never touched.

Numbers produced here are meaningless; the test checks that files, joins and metrics work.
Always run in a **fresh kernel**, top to bottom. Real data and real models: `test_experiments.ipynb`.


In [ ]:
from dotenv import load_dotenv
load_dotenv()  # Load environment variables from .env file
import json
import os
import shutil
import sys

# Test mode must be switched on before config is imported; config reads it only once per kernel.
os.environ["KUKI_TEST"] = "1"
import config

if getattr(config, "TEST_MODE", None) is not True:  # re-running this cell is fine; mixing with real data is not
    raise RuntimeError(f"config loaded in the wrong mode or from the wrong place ({config.__file__}): restart the kernel")

assert config.BASE_DIR == config.ROOT / "test_run"  # never delete anything outside test_run
os.chdir(config.ROOT)  # so that %run finds the scripts
shutil.rmtree(config.BASE_DIR, ignore_errors=True)

# Dummy codebook prompts; the real wrappers.json is reused.
config.PROMPT_DIR.mkdir(parents=True)
shutil.copy(config.ROOT / "prompts" / "wrappers.json", config.PROMPT_DIR)
for layer in ["L1", "L2", "L3", "L4"]:
    for lang in ["en", "ru", "tr"]:
        (config.PROMPT_DIR / f"codebook_{layer}_{lang}.md").write_text(f"Test codebook {layer} ({lang})", encoding="utf-8")

## 1. Synthetic aggregate file
Seven Russian articles in the format of `kuki_ru_aggregate.jsonl` (one record per article,
annotations keyed by Label Studio user id). Coverage mimics the block design: triple, double, single.


In [ ]:
def make_content(i):
    return (f"Газ и безопасность {i}\n\n"
            "Москва угрожает Европе прекращением поставок газа.\n\n"
            "Жители Киева пострадали от обстрелов.\n\n"
            "Эксперты предупреждают о кризисе.")


def span(content, text, layer, label, **fields):
    start = content.index(text)
    return {"layer": layer, "label": label, "start": start, "end": start + len(text), "text": text, **fields}


def annotation(letter, content, i):
    # Even articles: all annotators agree on 'Economic' and a 'Call' span -> human alpha is defined.
    frames = ["Economic"] if i % 2 == 0 else []
    spans = [span(content, "о кризисе", "l3_persuasion", "Call")] if i % 2 == 0 else []
    if letter == "A":
        frames += ["Security & defense", "Political"]
        spans += [span(content, "Москва", "l1_roles", "ANTAGONIST"),
                  span(content, "Европе", "l1_roles", "INNOCENT"),
                  span(content, "угрожает Европе", "l3_persuasion", "Manipulative Wording"),
                  span(content, "прекращением поставок газа", "l4_coded", "Coded Phrase",
                       literal="stopping gas", insider="energy as weapon", why="context")]
    elif letter == "B":
        frames += ["Security & defense"]
        spans += [span(content, "Москва", "l1_roles", "ANTAGONIST"),
                  span(content, "Жители Киева", "l1_roles", "INNOCENT"),
                  span(content, "Эксперты предупреждают", "l3_persuasion", "Justification")]
    else:
        frames += ["Security & defense", "External regulation & reputation"]
        spans += [span(content, "Европе", "l1_roles", "INNOCENT"),
                  span(content, "пострадали от обстрелов", "l3_persuasion", "Manipulative Wording")]
    return {"frames": frames, "spans": spans}


COVERAGE = ["ABC", "ABC", "AB", "BC", "A", "B", "C"]  # annotators of each article
USER_IDS = {letter: user for user, letter in config.ANNOTATOR_IDS["ru"].items()}
SOURCES = ["ria_novosti", "theinsider"]

config.DATA_DIR.mkdir(parents=True)
with open(config.AGGREGATES["ru"], "w", encoding="utf-8") as f:
    for i, letters in enumerate(COVERAGE):
        content = make_content(i)
        rec = {"doc_id": f"{SOURCES[i % 2]}:{i}", "lang": "ru", "source": SOURCES[i % 2],
               "title": content.split("\n")[0], "content": content, "n_annotators": len(letters),
               "annotations": {str(USER_IDS[l]): annotation(l, content, i) for l in letters}}
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")


## 2. Structural check
The same check as in `test_experiments.ipynb`.


In [ ]:
import json
from collections import Counter

from llm_io import read_jsonl


def inspect_aggregate(lang):
    """Structural check of an aggregate file: counts, users, span layers."""
    recs = read_jsonl(config.AGGREGATES[lang])
    anns = [a for r in recs for a in r["annotations"].values()]
    print(f"{config.AGGREGATES[lang].name}: {len(recs)} articles, {len(anns)} annotations")
    print("  articles by number of annotators:", dict(sorted(Counter(r["n_annotators"] for r in recs).items())))
    print("  users:", dict(Counter(u for r in recs for u in r["annotations"])), "expected:", config.ANNOTATOR_IDS[lang])
    print("  span layers:", dict(Counter(s["layer"] for a in anns for s in a["spans"])))
    print("  sources:", dict(Counter(r["source"] for r in recs)), "unknown:", {r["source"] for r in recs} - set(config.SOURCES))


inspect_aggregate("ru")


## 3. Data preparation and splits

In [ ]:
%run prep_00_load_data.py --langs ru

In [ ]:
from llm_io import read_jsonl

# Expected: the annotators per article as in COVERAGE.
for a in read_jsonl(config.PREP_DIR / "articles.jsonl"):
    print(a["article_id"], a["annotators"])


In [ ]:
import pandas as pd
import config
from llm_io import read_jsonl

articles = read_jsonl(config.PREP_DIR / "articles.jsonl")
print("entities of article 0:", articles[0]["entities"])
labels = pd.read_csv(config.PREP_DIR / "human_labels.csv")
# Expected: Москва/Европе/Жители Киева as entities; frames and paragraph labels per annotator.
labels[labels["value"] == 1].head(12)

In [ ]:
%run prep_01_make_splits.py --n-dev 1 --n-l4 2

## 4. Prompt check
What the model actually sees: one paragraph-with-context call, native prompt language.

In [ ]:
import llm_io

article = llm_io.load_split("s1_main_grid")[0]
for message in llm_io.build_messages(article, "L3", "para_ctx", "native", target=2):
    print(f"--- {message['role']} ---\n{message['content']}\n")
print(json.dumps(llm_io.output_schema("L1", llm_io.entity_names(article)), ensure_ascii=False)[:300])

## 5. Stage 1: main grid (mock models)
Two mock models give two sizes and two profiles, so the decomposition has something to split.

In [ ]:
%run s1_main_grid.py --model mock-M --dev
%run s1_main_grid.py --model mock-M
%run s1_main_grid.py --model mock-S

In [ ]:
# Re-running must resume: expect '0 to run'.
%run s1_main_grid.py --model mock-S

In [ ]:
# One prediction file per experiment and model.
print(sorted(p.name for p in config.PRED_DIR.iterdir()))
predictions = pd.DataFrame(read_jsonl(config.PRED_DIR / "s1_main_grid__mock-M.jsonl") + read_jsonl(config.PRED_DIR / "s1_main_grid__mock-S.jsonl"))
print(predictions.groupby(["model", "layer", "granularity", "prompt_lang"]).size().unstack())
predictions[["key", "raw", "parse_ok"]].head(3)

In [ ]:
%run evaluate.py --experiment s1_main_grid

Expected here: a `SingularMatrixWarning`, because in the mock grid size and profile are confounded (mock-S = S + english, mock-M = M + multilingual). The real grid crosses them.

In [ ]:
%run analyze_s1_decomposition.py

## 6. Stages 2-5 (mock models)

In [ ]:
%run s2_stability.py --model mock-M --granularity para_ctx --prompt-lang en --n-samples 3
%run evaluate.py --experiment s2_stability

In [ ]:
%run s3_metadata_probe.py --model mock-M
%run evaluate.py --experiment s3_metadata_probe
pd.read_csv(config.RESULT_DIR / "s3_metadata_probe_prevalence_by_meta.csv").head()

In [ ]:
%run s4_generalisation.py --model mock-S --granularity doc --prompt-lang en
%run evaluate.py --experiment s4_generalisation

In [ ]:
%run s5_l4_pilot.py --model mock-M
%run evaluate.py --experiment s5_l4_pilot

## 7. Smoke test with a real model
Use a **fresh kernel without the test cells above**, after `prep_00`/`prep_01` ran on the real
data, on the GPU machine. Loads the model with transformers and runs one constrained call:
check that the output is valid JSON, and look at token count and time per call.

In [ ]:
from tqdm.auto import tqdm

RUN_REAL = True
print("RUN_REAL is set to", RUN_REAL)
if RUN_REAL:
    import llm_io
    print("Loading model and running a smoke test on the first article of the dev split...")
    backend = llm_io.load_model("olmo3-7b")
    print("Model loaded. Running smoke test...")
    article = llm_io.load_split("dev")[0]
    jobs = list(llm_io.make_jobs("smoke", [article], "olmo3-7b", ["L2", "L3"], ["doc"], ["en"]))
    for job in tqdm(jobs, desc="Smoke test"):
        record = llm_io.call_llm(backend, job)
        tqdm.write(f"{job['layer']} {record['parsed']} | tokens in/out: {record['tokens_in']} {record['tokens_out']} "
                   f"| seconds: {record['seconds']} | error: {record['error']}")

In [ ]:
import time, torch, transformers
from transformers import AutoTokenizer, AutoModelForCausalLM
print(transformers.__version__, torch.cuda.is_available())

hf = "allenai/Olmo-3-7B-Instruct"
t = time.time(); tok = AutoTokenizer.from_pretrained(hf); print("tokenizer", time.time() - t)
t = time.time(); llm = AutoModelForCausalLM.from_pretrained(hf, dtype="auto", device_map="auto"); print("model", time.time() - t)
t = time.time(); data = build_token_enforcer_tokenizer_data(tok); print("enforcer", time.time() - t)